# CS 435 — Project 3: Medical X-Ray Anomaly Detection
## Phase 5: Final Comparison, Analysis & Report Figures

**Goal of this notebook:**  
Load the saved metrics from Phase 2 (CNN baseline) and Phase 4 (GAN-augmented CNN), produce all final comparison figures and tables, interpret the results, and export everything needed for the project report.

### The central research question this project answers:
> *Does augmenting a multi-label chest X-ray classifier's training set with GAN-synthesized minority-class images improve detection performance on those minority classes, measured by AUC-ROC and F1-score on a held-out real test set?*

Phase 5 is where we answer that question with evidence.

**Prerequisites:** Phases 1–4 must have been run and all CSVs saved to Google Drive.

---
## Step 1: Mount Drive and import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         11,
})

print("Libraries loaded.")

---
## Step 2: Configuration and paths

In [ ]:
BASE_DIR    = os.path.abspath("")
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
SPLITS_DIR  = os.path.join(BASE_DIR, 'splits')
REPORT_DIR  = os.path.join(BASE_DIR, 'report_figures')
os.makedirs(REPORT_DIR, exist_ok=True)

TARGET_CLASSES   = ['No Finding', 'Atelectasis', 'Effusion', 'Cardiomegaly', 'Pneumonia']
MINORITY_CLASSES = ['Pneumonia', 'Cardiomegaly']

COLOR_BASELINE  = '#1565C0'
COLOR_AUGMENTED = '#2E7D32'
COLOR_DELTA_POS = '#43A047'
COLOR_DELTA_NEG = '#E53935'

print("Configuration ready.")
print(f"Report figures will save to: {REPORT_DIR}")

---
## Step 3: Load metrics from Phase 2 and Phase 4

Both CSV files were saved with the same column schema:
`class | auc_roc | f1 | precision | recall | model`

We merge them into a single wide DataFrame for easy comparison.

In [ ]:
baseline_path  = os.path.join(RESULTS_DIR, 'baseline_metrics.csv')
augmented_path = os.path.join(RESULTS_DIR, 'augmented_metrics.csv')

baseline_df  = pd.read_csv(baseline_path)
augmented_df = pd.read_csv(augmented_path)

# Merge on class
comparison_df = baseline_df.merge(augmented_df, on='class',
                                   suffixes=('_baseline', '_augmented'))

# Compute improvement deltas
comparison_df['auc_delta']  = comparison_df['auc_roc_augmented']   - comparison_df['auc_roc_baseline']
comparison_df['f1_delta']   = comparison_df['f1_augmented']        - comparison_df['f1_baseline']
comparison_df['prec_delta'] = comparison_df['precision_augmented'] - comparison_df['precision_baseline']
comparison_df['rec_delta']  = comparison_df['recall_augmented']    - comparison_df['recall_baseline']

print("Metrics loaded and merged.")
print("\nFull comparison table:")
display_cols = ['class',
                'auc_roc_baseline', 'auc_roc_augmented', 'auc_delta',
                'f1_baseline',      'f1_augmented',      'f1_delta']
print(comparison_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))

---
## Step 4: Figure 1 — AUC-ROC Grouped Bar Chart

Side-by-side bars for each class: baseline (blue) vs augmented (green). This is the primary result figure for the report — it directly shows which classes improved.

In [ ]:
# Use only per-class rows (exclude MACRO_AVG for this chart)
class_df = comparison_df[comparison_df['class'] != 'MACRO_AVG'].copy()
macro_df = comparison_df[comparison_df['class'] == 'MACRO_AVG'].copy()

x = np.arange(len(TARGET_CLASSES))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - w/2,
               class_df.set_index('class').loc[TARGET_CLASSES, 'auc_roc_baseline'],
               w, label='Baseline CNN (Phase 2)',
               color=COLOR_BASELINE, alpha=0.85, edgecolor='white')

bars2 = ax.bar(x + w/2,
               class_df.set_index('class').loc[TARGET_CLASSES, 'auc_roc_augmented'],
               w, label='GAN-Augmented CNN (Phase 4)',
               color=COLOR_AUGMENTED, alpha=0.85, edgecolor='white')

# Annotate bars with AUC values
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9,
            color=COLOR_BASELINE, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9,
            color=COLOR_AUGMENTED, fontweight='bold')

# Annotate minority classes with a marker
for i, cls in enumerate(TARGET_CLASSES):
    if cls in MINORITY_CLASSES:
        ax.annotate('★ GAN augmented', xy=(x[i], 0.52),
                    ha='center', fontsize=8, color='#FF6F00',
                    fontweight='bold')

# Macro average reference line
macro_base = macro_df['auc_roc_baseline'].values[0]
macro_aug  = macro_df['auc_roc_augmented'].values[0]
ax.axhline(macro_base, color=COLOR_BASELINE, linestyle=':', linewidth=1.5, alpha=0.6,
           label=f'Baseline macro avg: {macro_base:.3f}')
ax.axhline(macro_aug,  color=COLOR_AUGMENTED, linestyle=':', linewidth=1.5, alpha=0.6,
           label=f'Augmented macro avg: {macro_aug:.3f}')

ax.set_xticks(x)
ax.set_xticklabels(TARGET_CLASSES, fontsize=11)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.set_title('AUC-ROC per Class: Baseline CNN vs GAN-Augmented CNN\n'
             '(Test set — real NIH X-rays only)', fontsize=13)
ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig1_auc_comparison_bars.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 5: Figure 2 — F1-Score Grouped Bar Chart

AUC measures ranking quality; F1 measures precision-recall balance at a fixed threshold. Both matter — AUC tells us about the model's discriminative power, F1 tells us about practical prediction quality.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - w/2,
               class_df.set_index('class').loc[TARGET_CLASSES, 'f1_baseline'],
               w, label='Baseline CNN', color=COLOR_BASELINE, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2,
               class_df.set_index('class').loc[TARGET_CLASSES, 'f1_augmented'],
               w, label='GAN-Augmented CNN', color=COLOR_AUGMENTED, alpha=0.85, edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9,
            color=COLOR_BASELINE, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9,
            color=COLOR_AUGMENTED, fontweight='bold')

for i, cls in enumerate(TARGET_CLASSES):
    if cls in MINORITY_CLASSES:
        ax.annotate('★ GAN augmented', xy=(x[i], 0.02),
                    ha='center', fontsize=8, color='#FF6F00', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(TARGET_CLASSES, fontsize=11)
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_ylim(0.0, 1.0)
ax.set_title('F1-Score per Class: Baseline CNN vs GAN-Augmented CNN\n'
             '(Test set — real NIH X-rays only)', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig2_f1_comparison_bars.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 6: Figure 3 — Delta Heatmap (Improvement Map)

This heatmap shows exactly how much each metric changed per class due to GAN augmentation. Green = improvement, red = regression. This is one of the most informative single figures for the report — it communicates the effect of augmentation at a glance.

In [ ]:
delta_data = class_df.set_index('class').loc[
    TARGET_CLASSES,
    ['auc_delta', 'f1_delta', 'prec_delta', 'rec_delta']
].rename(columns={
    'auc_delta':  'AUC-ROC Δ',
    'f1_delta':   'F1 Δ',
    'prec_delta': 'Precision Δ',
    'rec_delta':  'Recall Δ'
})

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    delta_data.T,
    annot=True, fmt='.4f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Augmented − Baseline (positive = improvement)'},
    ax=ax
)

ax.set_title('Metric Improvement Heatmap: GAN Augmentation Effect\n'
             '(Augmented CNN − Baseline CNN, per class)', fontsize=13)
ax.set_xlabel('Disease Class', fontsize=11)
ax.set_ylabel('Metric', fontsize=11)

# Highlight the minority class columns
for i, cls in enumerate(TARGET_CLASSES):
    if cls in MINORITY_CLASSES:
        ax.add_patch(plt.Rectangle((i, 0), 1, len(delta_data.columns),
                                   fill=False, edgecolor='#FF6F00',
                                   lw=2.5, clip_on=False))

plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig3_delta_heatmap.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")
print("Orange border = GAN-augmented minority classes")

---
## Step 7: Figure 4 — Training Curve Comparison (Loss + AUC)

Loaded from the CSVLogger files saved during training in Phases 2 and 4.

In [ ]:
base_log = pd.read_csv(os.path.join(RESULTS_DIR, 'baseline_training_log.csv'))
aug_log  = pd.read_csv(os.path.join(RESULTS_DIR, 'augmented_training_log.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Validation Loss ────────────────────────────────────────────────────────
axes[0].plot(base_log['val_loss'], color=COLOR_BASELINE,  lw=2,
             label='Baseline val loss')
axes[0].plot(aug_log['val_loss'],  color=COLOR_AUGMENTED, lw=2,
             label='Augmented val loss')
axes[0].plot(base_log['loss'],     color=COLOR_BASELINE,  lw=1,
             linestyle='--', alpha=0.5, label='Baseline train loss')
axes[0].plot(aug_log['loss'],      color=COLOR_AUGMENTED, lw=1,
             linestyle='--', alpha=0.5, label='Augmented train loss')
axes[0].set_title('Training & Validation Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend(fontsize=8)

# ── Validation AUC ────────────────────────────────────────────────────────
axes[1].plot(base_log['val_auc'], color=COLOR_BASELINE,  lw=2,
             label='Baseline val AUC')
axes[1].plot(aug_log['val_auc'],  color=COLOR_AUGMENTED, lw=2,
             label='Augmented val AUC')
axes[1].plot(base_log['auc'],     color=COLOR_BASELINE,  lw=1,
             linestyle='--', alpha=0.5, label='Baseline train AUC')
axes[1].plot(aug_log['auc'],      color=COLOR_AUGMENTED, lw=1,
             linestyle='--', alpha=0.5, label='Augmented train AUC')
axes[1].set_title('Training & Validation AUC-ROC', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC-ROC')
axes[1].legend(fontsize=8)

plt.suptitle('Phase 2 Baseline vs Phase 4 GAN-Augmented CNN — Training History',
             fontsize=13, y=1.02)
plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig4_training_curves_comparison.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 8: Figure 5 — Precision–Recall Comparison (4-metric grid)

A 2×2 subplot grid showing all four metrics (AUC, F1, Precision, Recall) side by side, so the reader can see the full picture in one view.

In [ ]:
metrics_map = [
    ('auc_roc_baseline',   'auc_roc_augmented',   'AUC-ROC'),
    ('f1_baseline',        'f1_augmented',         'F1-Score'),
    ('precision_baseline', 'precision_augmented',  'Precision'),
    ('recall_baseline',    'recall_augmented',     'Recall'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (col_base, col_aug, title) in zip(axes, metrics_map):
    cdf = class_df.set_index('class').loc[TARGET_CLASSES]
    xpos = np.arange(len(TARGET_CLASSES))

    b = ax.bar(xpos - 0.2, cdf[col_base], 0.38,
               label='Baseline', color=COLOR_BASELINE, alpha=0.85, edgecolor='white')
    a = ax.bar(xpos + 0.2, cdf[col_aug],  0.38,
               label='Augmented', color=COLOR_AUGMENTED, alpha=0.85, edgecolor='white')

    ax.set_xticks(xpos)
    ax.set_xticklabels(TARGET_CLASSES, fontsize=9, rotation=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

    # Highlight minority class bars with an orange outline
    for i, cls in enumerate(TARGET_CLASSES):
        if cls in MINORITY_CLASSES:
            ax.add_patch(plt.Rectangle((i - 0.4, 0), 0.8, 1.05,
                                       fill=False, edgecolor='#FF6F00',
                                       lw=1.5, linestyle='--', alpha=0.6,
                                       clip_on=True))

plt.suptitle('All Metrics: Baseline vs GAN-Augmented CNN\n'
             '(Dashed orange = GAN-augmented minority classes)',
             fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig5_all_metrics_grid.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 9: Table 1 — Full Metrics Table (report-ready)

A clean, formatted DataFrame showing all metrics for both models plus the delta column. This is the primary results table for the report — copy this directly.

In [ ]:
report_table = comparison_df[[
    'class',
    'auc_roc_baseline', 'auc_roc_augmented', 'auc_delta',
    'f1_baseline',      'f1_augmented',      'f1_delta',
    'precision_baseline', 'precision_augmented',
    'recall_baseline',    'recall_augmented',
]].copy()

# Rename for clean presentation
report_table.columns = [
    'Class',
    'AUC (Base)', 'AUC (Aug)', 'ΔAUC',
    'F1 (Base)',  'F1 (Aug)',  'ΔF1',
    'Prec (Base)', 'Prec (Aug)',
    'Rec (Base)',  'Rec (Aug)',
]

# Round all numeric columns
num_cols = report_table.columns[1:]
report_table[num_cols] = report_table[num_cols].round(4)

print("=" * 100)
print("TABLE 1: Full Results — Baseline CNN vs GAN-Augmented CNN (Test Set)")
print("=" * 100)
print(report_table.to_string(index=False))
print()
print("Note: Positive ΔAUC and ΔF1 indicate improvement from GAN augmentation.")
print("★ Pneumonia and Cardiomegaly are the GAN-augmented minority classes.")

# Save as CSV for easy copy-paste into report
table_path = os.path.join(REPORT_DIR, 'table1_full_results.csv')
report_table.to_csv(table_path, index=False)
print(f"\nSaved to: {table_path}")

---
## Step 10: Figure 6 — Class Distribution (before vs after augmentation)

Reproduces the key data motivation figure — shows the class imbalance problem and how the GAN addressed it.

In [ ]:
train_orig = pd.read_csv(os.path.join(SPLITS_DIR, 'train.csv'))
train_aug  = pd.read_csv(os.path.join(SPLITS_DIR, 'train_augmented.csv'))

orig_counts = train_orig[TARGET_CLASSES].sum().reindex(TARGET_CLASSES)
aug_counts  = train_aug[TARGET_CLASSES].sum().reindex(TARGET_CLASSES)

xpos = np.arange(len(TARGET_CLASSES))
fig, ax = plt.subplots(figsize=(12, 5))

bars1 = ax.bar(xpos - 0.2, orig_counts, 0.38, label='Original (real only)',
               color='#90CAF9', edgecolor='white')
bars2 = ax.bar(xpos + 0.2, aug_counts,  0.38, label='Augmented (real + GAN synthetic)',
               color=COLOR_AUGMENTED, alpha=0.85, edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(xpos)
ax.set_xticklabels(TARGET_CLASSES, fontsize=11)
ax.set_ylabel('Number of training images')
ax.set_title('Training Set Class Distribution: Before and After GAN Augmentation',
             fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig6_class_distribution_comparison.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 11: Results Interpretation

This cell generates a structured written interpretation of the results — ready to paste directly into your project report discussion section.

In [ ]:
# Extract key numbers for the interpretation
macro_base_auc = macro_df['auc_roc_baseline'].values[0]
macro_aug_auc  = macro_df['auc_roc_augmented'].values[0]
macro_delta    = macro_df['auc_delta'].values[0]

rows = class_df.set_index('class')

print("=" * 80)
print("RESULTS INTERPRETATION — CS 435 Project 3")
print("=" * 80)

print("\n--- MACRO AVERAGE ---")
print(f"  Baseline macro AUC  : {macro_base_auc:.4f}")
print(f"  Augmented macro AUC : {macro_aug_auc:.4f}")
direction = 'improved' if macro_delta > 0 else 'declined'
print(f"  Overall macro AUC {direction} by {abs(macro_delta):.4f} ({abs(macro_delta)*100:.2f}pp)")

print("\n--- MINORITY CLASSES (GAN-augmented) ---")
for cls in MINORITY_CLASSES:
    base_auc = rows.loc[cls, 'auc_roc_baseline']
    aug_auc  = rows.loc[cls, 'auc_roc_augmented']
    delta    = rows.loc[cls, 'auc_delta']
    base_f1  = rows.loc[cls, 'f1_baseline']
    aug_f1   = rows.loc[cls, 'f1_augmented']
    f1_delta = rows.loc[cls, 'f1_delta']
    dir_auc  = '↑ improved' if delta > 0 else '↓ declined'
    dir_f1   = '↑ improved' if f1_delta > 0 else '↓ declined'
    print(f"\n  {cls}:")
    print(f"    AUC:  {base_auc:.4f} → {aug_auc:.4f}  ({dir_auc} by {abs(delta):.4f})")
    print(f"    F1:   {base_f1:.4f} → {aug_f1:.4f}  ({dir_f1} by {abs(f1_delta):.4f})")

print("\n--- MAJORITY CLASSES (not augmented) ---")
majority_classes = [c for c in TARGET_CLASSES if c not in MINORITY_CLASSES]
for cls in majority_classes:
    delta = rows.loc[cls, 'auc_delta']
    direction = '↑' if delta > 0 else '↓'
    print(f"  {cls}: AUC delta = {direction}{abs(delta):.4f} (expected: near zero)")

print("\n--- CONCLUSION ---")
minority_improved = all(
    rows.loc[cls, 'auc_delta'] > 0 for cls in MINORITY_CLASSES
)
if minority_improved:
    print("  ✓ GAN augmentation IMPROVED AUC for ALL minority classes.")
    print("  The hypothesis is SUPPORTED: synthetic GAN images reduced the effect")
    print("  of class imbalance and helped the CNN learn better representations")
    print("  for Pneumonia and Cardiomegaly.")
else:
    print("  ✗ GAN augmentation did NOT improve AUC for all minority classes.")
    print("  Possible reasons: insufficient GAN training epochs, mode collapse,")
    print("  or too few synthetic images. See Phase 3 troubleshooting notes.")

print("\n" + "=" * 80)

---
## Step 12: Figure 7 — Summary Scorecard

A clean, single-figure summary suitable for a presentation slide or the executive summary section of the report.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# Build table data
table_data = []
col_labels = ['Class', 'AUC (Base)', 'AUC (Aug)', 'ΔAUC', 'F1 (Base)', 'F1 (Aug)', 'ΔF1']

all_rows = TARGET_CLASSES + ['MACRO_AVG']
for cls in all_rows:
    r = comparison_df[comparison_df['class'] == cls].iloc[0]
    label = cls if cls != 'MACRO_AVG' else 'Macro Avg'
    table_data.append([
        label,
        f"{r['auc_roc_baseline']:.4f}",
        f"{r['auc_roc_augmented']:.4f}",
        f"{r['auc_delta']:+.4f}",
        f"{r['f1_baseline']:.4f}",
        f"{r['f1_augmented']:.4f}",
        f"{r['f1_delta']:+.4f}",
    ])

tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)

# Color the header row
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#1565C0')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Color delta cells: green if positive, red if negative
for i, cls in enumerate(all_rows, start=1):
    r = comparison_df[comparison_df['class'] == cls].iloc[0]
    # ΔAUC is column 3, ΔF1 is column 6
    for col_idx, delta_key in [(3, 'auc_delta'), (6, 'f1_delta')]:
        val = r[delta_key]
        color = '#C8E6C9' if val > 0 else ('#FFCDD2' if val < 0 else '#FFFFFF')
        tbl[(i, col_idx)].set_facecolor(color)
    # Highlight minority class rows
    if cls in MINORITY_CLASSES:
        tbl[(i, 0)].set_facecolor('#FFF9C4')
        tbl[(i, 0)].set_text_props(fontweight='bold')

ax.set_title('Project 3 Results Summary: CNN Baseline vs GAN-Augmented CNN\n'
             '(Green = improvement | Yellow rows = GAN-augmented classes)',
             fontsize=12, pad=20)

plt.tight_layout()
path = os.path.join(REPORT_DIR, 'fig7_results_scorecard.png')
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

---
## Step 13: List all saved report figures

In [ ]:
print("=" * 60)
print("ALL REPORT-READY FILES:")
print("=" * 60)
files = sorted(os.listdir(REPORT_DIR))
for f in files:
    fpath = os.path.join(REPORT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:55s} {size_kb:6.1f} KB")

print("\n" + "=" * 60)
print("FIGURE GUIDE FOR REPORT:")
print("=" * 60)
guide = [
    ("fig1_auc_comparison_bars.png",     "Primary results — AUC per class, grouped bars"),
    ("fig2_f1_comparison_bars.png",      "F1-score per class, grouped bars"),
    ("fig3_delta_heatmap.png",           "Improvement heatmap — all metrics, all classes"),
    ("fig4_training_curves_comparison.png", "Loss + AUC training curves, both models"),
    ("fig5_all_metrics_grid.png",        "4-metric grid — AUC, F1, Precision, Recall"),
    ("fig6_class_distribution_comparison.png", "Data: before/after GAN augmentation"),
    ("fig7_results_scorecard.png",       "Summary table with color-coded deltas"),
    ("table1_full_results.csv",          "Full results table — paste into report"),
]
for fname, description in guide:
    print(f"  {fname:50s} → {description}")

---
## Phase 5 Summary — Project Complete

### What this project demonstrated:

This project built a complete deep learning pipeline for medical image analysis, spanning five phases:

| Phase | What was built | Key output |
|-------|---------------|------------|
| 1 | NIH Chest X-Ray14 download, preprocessing, class imbalance analysis, 70/15/15 split | `train/val/test.csv`, class distribution chart |
| 2 | CNN baseline (Conv2D × 3 → GAP → sigmoid), trained on real data only | `baseline_metrics.csv`, ROC curves, training logs |
| 3 | DCGAN per minority class (Conv2DTranspose generator, Conv2D discriminator), GAN-synthesized X-rays | `train_augmented.csv`, synthetic PNG images |
| 4 | Same CNN retrained on real + GAN-synthetic data | `augmented_metrics.csv`, comparison training curves |
| 5 | Statistical comparison, 7 report figures, results interpretation | All `report_figures/` PNG files + summary table |

### Connection to course textbooks:
- **Chollet** (*Deep Learning with Python*): CNN architecture patterns (Ch. 8–9), multi-label sigmoid output (Ch. 14), AUC-ROC for imbalanced data (Ch. 6), callbacks EarlyStopping + ModelCheckpoint (Ch. 7)
- **Rosebrock** (*Deep Learning for Computer Vision*): Conv2D feature extraction hierarchy, data augmentation as regularization (Ch. 2–5)
- **Brownlee** (*GANs with Python*): DCGAN architecture best practices (Ch. 5), Conv2DTranspose upsampling (Ch. 3), adversarial training loop with `train_on_batch` (Ch. 6–7)

### For your final report discussion:
Use `fig1` as your primary result figure. Use `fig3` (the delta heatmap) as evidence that the GAN selectively improved the minority classes. Use `fig6` to motivate the problem (class imbalance). Use `fig7` as your executive summary table.

All figures are saved to `CS435_Project3/report_figures/` on your Google Drive.